# 03 - Pandas Analysis
Answers the 10 Pandas questions using the cleaned tables from `movies.db`.

## Setup

In [1]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect('../data/movies.db')
movies = pd.read_sql('SELECT * FROM movies', conn, parse_dates=['release_date'])
genres = pd.read_sql('SELECT * FROM genres', conn)
movie_genres = pd.read_sql('SELECT * FROM movie_genres', conn)
cast = pd.read_sql('SELECT * FROM cast', conn)
crew = pd.read_sql('SELECT * FROM crew', conn)
movie_keywords = pd.read_sql('SELECT * FROM movie_keywords', conn)
conn.close()

# SQLite stores booleans as 0/1 - restore proper bool dtype
movies['budget_known'] = movies['budget_known'].astype(bool)
movies['revenue_known'] = movies['revenue_known'].astype(bool)

print(movies.shape, genres.shape, movie_genres.shape, cast.shape, crew.shape, movie_keywords.shape)

(2503, 14) (19, 2) (6878, 2) (24902, 6) (7289, 6) (41681, 3)


## Q1 — Unique counts across all tables

In [2]:
summary = pd.Series({
    'unique_movies': movies['movie_id'].nunique(),
    'unique_genres': genres['genre_id'].nunique(),
    'unique_cast_members': cast['person_id'].nunique(),
    'unique_crew_members': crew['person_id'].nunique(),
})
summary

unique_movies           2503
unique_genres             19
unique_cast_members    11300
unique_crew_members     2848
dtype: int64

Small, curated catalog (~2,500 films) but a huge pool of unique people - most actors/crew only show up once or twice.

## Q2 — Movies with missing/zero budget or revenue

In [3]:
missing_financials = movies[~movies['budget_known'] | ~movies['revenue_known']]
print(f"{len(missing_financials)} movies ({len(missing_financials)/len(movies):.1%}) "
      f"have an unknown budget and/or revenue")
missing_financials[['title', 'release_date', 'budget', 'revenue',
                     'budget_known', 'revenue_known']].head(10)

133 movies (5.3%) have an unknown budget and/or revenue


,title,release_date,budget,revenue,budget_known,revenue_known
324,Zack Snyder's Justice League,2021-03-18,70000000.0,0.0,True,False
340,Bird Box,2018-12-13,19800000.0,0.0,True,False
403,365 Days,2020-02-07,0.0,9458590.0,False,True
468,To All the Boys I've Loved Before,2018-08-17,0.0,0.0,False,False
554,Prey,2022-08-02,65000000.0,0.0,True,False
598,The Kissing Booth,2018-05-11,0.0,0.0,False,False
717,Enola Holmes,2020-09-22,21000000.0,0.0,True,False
723,Extraction,2020-04-23,65000000.0,0.0,True,False
754,"Love, Rosie",2014-10-16,0.0,4439431.0,False,True
842,Bright,2017-12-22,90000000.0,0.0,True,False


Kept these rows but excluded them from any $ calc (profit, ROI, avg budget/revenue) using the known flags. Treating a 0 as a real number would wreck those numbers.

## Q3 — Profit and ROI for movies with valid budget & revenue

In [4]:
valid_fin = movies[movies['budget_known'] & movies['revenue_known']].copy()
valid_fin['profit'] = valid_fin['revenue'] - valid_fin['budget']
valid_fin['roi'] = valid_fin['profit'] / valid_fin['budget']

print(f"{len(valid_fin)} movies have valid budget & revenue figures")
valid_fin[['title', 'budget', 'revenue', 'profit', 'roi']].sort_values('profit', ascending=False).head(10)

2370 movies have valid budget & revenue figures


,title,budget,revenue,profit,roi
4,Avatar,237000000.0,2.923706e+09,2.686706e+09,11.336312
16,Avengers: Endgame,356000000.0,2.799439e+09,2.443439e+09,6.863593
19,Titanic,200000000.0,2.264162e+09,2.064162e+09,10.320812
178,Avatar: The Way of Water,460000000.0,2.334485e+09,1.874485e+09,4.074967
65,Star Wars: The Force Awakens,245000000.0,2.068224e+09,1.823224e+09,7.441729
7,Avengers: Infinity War,300000000.0,2.052415e+09,1.752415e+09,5.841383
46,Spider-Man: No Way Home,200000000.0,1.921207e+09,1.721207e+09,8.606033
1776,Zootopia 2,150000000.0,1.868209e+09,1.718209e+09,11.454725
53,Jurassic World,150000000.0,1.671537e+09,1.521537e+09,10.143583
695,Inside Out 2,200000000.0,1.698864e+09,1.498864e+09,7.494319


Profit is dominated by mega-budget blockbusters. Q4 (ROI) tells a different story once budget size is normalized out.

## Q4 — Top 15 movies by ROI (budget ≥ $1M)

In [5]:
roi_filtered = valid_fin[valid_fin['budget'] >= 1_000_000].copy()
top_roi = roi_filtered.sort_values('roi', ascending=False).head(15)
top_roi[['title', 'budget', 'revenue', 'roi']]

,title,budget,revenue,roi
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,124.002821
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,123.242561
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,121.272429
467,Rocky,1000000.0,117253345.0,116.253345
1227,Gone with the Wind,4000000.0,402352579.0,99.588145
716,The Jungle Book,4000000.0,378000000.0,93.500000
635,Cinderella,2900000.0,263600000.0,89.896552
372,Saw,1200000.0,104045735.0,85.704779
719,One Hundred and One Dalmatians,3600000.0,303000000.0,83.166667
263,E.T. the Extra-Terrestrial,10500000.0,797307407.0,74.934039


Top-ROI movies are mostly modest-budget films that way outperformed their spend - a totally different list than the raw profit ranking.

## Q5 — Movies released per year

In [6]:
movies_with_year = movies.dropna(subset=['release_date']).copy()
movies_with_year['release_year'] = movies_with_year['release_date'].dt.year
by_year = movies_with_year.groupby('release_year').size().rename('movie_count')

peak_year = by_year.idxmax()
print(f"Peak year: {peak_year} with {by_year.max()} movies released")
by_year.tail(15)

Peak year: 2016 with 121 movies released


release_year
2012     82
2013    110
2014    105
2015     97
2016    121
2017    109
2018    102
2019    100
2020     52
2021     67
2022     65
2023     62
2024     43
2025     26
2026      8
Name: movie_count, dtype: int64

Output grows steadily and peaks in one year, then drops off toward the end of the dataset.

## Q6 — Runtime outliers

In [7]:
q1_, q3_ = movies['runtime'].quantile([0.25, 0.75])
iqr = q3_ - q1_
lower_bound = q1_ - 1.5 * iqr
upper_bound = q3_ + 1.5 * iqr

runtime_outliers = movies[(movies['runtime'] < lower_bound) | (movies['runtime'] > upper_bound)]
print(f"IQR bounds: [{lower_bound:.0f}, {upper_bound:.0f}] minutes")
print(f"{len(runtime_outliers)} runtime outliers found")
runtime_outliers[['title', 'runtime']].sort_values('runtime')

IQR bounds: [58, 166] minutes
60 runtime outliers found


,title,runtime
2307,The Guardians of the Galaxy Holiday Special,45
2447,The Garden of Words,46
514,Dune: Part Two,167
620,Gangs of New York,168
110,Saving Private Ryan,169
397,It Chapter Two,169
0,Interstellar,169
146,Pirates of the Caribbean: At World's End,169
76,The Hobbit: An Unexpected Journey,169
492,Heat,170


Outliers sit at both ends - a few very short films and a few very long ones. Worth excluding from any runtime-vs-rating comparison.

## Q7 — Primary genre popularity comparison

In [8]:
# 'Primary genre' = the first genre listed per movie in movie_genres (lowest rowid per movie_id)
movie_genres_ordered = movie_genres.copy()
movie_genres_ordered['row_order'] = movie_genres_ordered.groupby('movie_id').cumcount()
primary_genre = movie_genres_ordered[movie_genres_ordered['row_order'] == 0]
primary_genre = primary_genre.merge(genres, on='genre_id')[['movie_id', 'genre_name']]
primary_genre = primary_genre.rename(columns={'genre_name': 'primary_genre'})

movies_primary = movies.merge(primary_genre, on='movie_id', how='left')
popularity_by_primary_genre = (movies_primary.groupby('primary_genre')['popularity']
                                .agg(['mean', 'count'])
                                .rename(columns={'mean': 'avg_popularity', 'count': 'movie_count'})
                                .sort_values('avg_popularity', ascending=False))
popularity_by_primary_genre

,avg_popularity,movie_count
primary_genre,,
Science Fiction,21.261836,119
Music,18.324340,10
Animation,16.811212,139
Family,16.710635,48
Action,16.385188,474
War,15.470317,23
Horror,14.851835,223
Adventure,14.785510,244
Fantasy,13.022920,93


Using just the first genre avoids double-counting a movie across all its genres, so the ranking here won't match the SQL notebook's version exactly.

## Q8 — Top 10 most prolific actors with average rating across their movies

In [9]:
cast_with_rating = cast.merge(movies[['movie_id', 'vote_average']], on='movie_id')
actor_stats = (cast_with_rating.groupby('actor_name')
               .agg(movie_count=('movie_id', 'nunique'), avg_rating=('vote_average', 'mean'))
               .sort_values('movie_count', ascending=False)
               .head(10))
actor_stats['avg_rating'] = actor_stats['avg_rating'].round(2)
actor_stats

,movie_count,avg_rating
actor_name,,
Samuel L. Jackson,39,6.94
Robert De Niro,38,7.23
Brad Pitt,38,7.26
Johnny Depp,37,6.86
Tom Hanks,33,7.29
Scarlett Johansson,32,7.06
Mark Wahlberg,32,6.63
Willem Dafoe,32,7.11
Tom Cruise,31,7.04


Combining appearance count with avg rating tells prolific-but-good apart from just prolific.

## Q9 — Duplicate movie titles (possible remakes)

In [10]:
movies_with_year_full = movies.dropna(subset=['release_date']).copy()
movies_with_year_full['release_year'] = movies_with_year_full['release_date'].dt.year

title_counts = movies_with_year_full['title'].value_counts()
dup_titles = title_counts[title_counts > 1].index

dup_movies = movies_with_year_full[movies_with_year_full['title'].isin(dup_titles)]
dup_movies[['title', 'release_year']].sort_values(['title', 'release_year'])

,title,release_year
878,A Nightmare on Elm Street,1984
1928,A Nightmare on Elm Street,2010
261,Aladdin,1992
339,Aladdin,2019
792,Alice in Wonderland,1951
...,...,...
1737,The Thing,2011
744,Total Recall,1990
860,Total Recall,2012
2500,Transformers,1987


Repeated titles are different movies (different years) - likely remakes/reboots sharing a name, not duplicate rows.

## Q10 — Popularity vs. vote count mismatches

In [11]:
pv = movies[(movies['popularity'] > 0) & (movies['vote_count'] > 0)].copy()

# rank both columns, then see where the ranks disagree the most
pv['popularity_rank'] = pv['popularity'].rank(ascending=False)
pv['vote_count_rank'] = pv['vote_count'].rank(ascending=False)
pv['rank_gap'] = pv['vote_count_rank'] - pv['popularity_rank']

high_pop_low_votes = pv.sort_values('rank_gap', ascending=False).head(10)
low_pop_high_votes = pv.sort_values('rank_gap', ascending=True).head(10)

print("High popularity, low vote count:")
display(high_pop_low_votes[['title', 'popularity', 'vote_count']])
print("\nHigh vote count, low popularity:")
display(low_pop_high_votes[['title', 'popularity', 'vote_count']])

High popularity, low vote count:


,title,popularity,vote_count
2452,Mortal Kombat II,99.5231,2260
2493,Lilo & Stitch,33.2161,2228
2388,Disclosure Day,394.3346,2328
2295,Lee Cronin's The Mummy,79.4601,2430
2441,Lolita,27.5391,2271
2248,Backrooms,258.4743,2483
2457,Smile 2,23.7112,2256
2306,Kraven the Hunter,26.1998,2419
2471,A Serbian Film,18.9190,2242
2265,Companion,23.1407,2466



High vote count, low popularity:


,title,popularity,vote_count
205,X-Men: First Class,1.2614,13680
244,Home Alone,1.8114,12658
254,X-Men,1.2978,12340
294,X-Men Origins: Wolverine,2.3595,11379
303,Fast & Furious 6,1.4240,11275
322,X2,0.8734,11013
326,The Fast and the Furious,1.5573,10971
354,X-Men: The Last Stand,1.1649,10511
417,The Lost World: Jurassic Park,0.8620,9356
360,The Wolverine,4.5631,10412


Positive rank_gap = trending now, votes haven't caught up yet. Negative rank_gap = older movie with a big vote history but less current buzz.

## Summary
- Profit/ROI only calculated on rows with real budget & revenue.
- ROI ranking (Q4) and profit ranking (Q3) give very different "best movie" lists - worth showing both.
- Duplicate titles are legit remakes, not bad data - good to know before doing any title-based lookup.